# Workshop 05: Regression, Lagged Predictors & Forecast Combinations
**Machine Learning II: Forecasting Economic Time Series** — Lazarski University  
**Academic Year:** 2026/2027 | **Format:** Python 3.10+ Scientific Stack (`pandas`, `statsmodels`, `scipy`, `sklearn`)

---

## 📋 Overview & Agenda
In this workshop, you will implement multivariate regression forecasters with deterministic trends and Fourier harmonics, build leak-free dynamic lag matrices, audit residual autocorrelation diagnostics, and implement forecast ensembles that illustrate the famous **Forecast Combination Puzzle**.

### Tasks Overview:
1. **Task 1: Deterministic Trends & Fourier Harmonics (RSAFS Retail Sales)**
2. **Task 2: Dynamic Lag Matrix Construction & The Leak-Audit Assertion**
3. **Task 3: Dynamic Regression & Residual Diagnostics (CPI & Oil)**
4. **Task 4: Ensembling & The Forecast Combination Puzzle under Walk-Forward Validation**
5. **Task 5: Written Responses to Syllabus Technical Questions**


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Set random seed for determinism
np.random.seed(42)

def get_data_dir():
    for candidate in [
        Path('data'),
        Path('../data'),
        Path('workshop_05_regression/data'),
        Path(__file__).resolve().parent.parent / 'data' if '__file__' in globals() else None,
    ]:
        if candidate and candidate.exists() and (candidate / 'CPIAUCSL.csv').exists():
            return candidate
    return Path('workshop_05_regression/data')

DATA_DIR = get_data_dir()
print(f"Data directory resolved to: {DATA_DIR.resolve()}")


## 📂 Task 1: Macroeconomic Data Ingestion
Implement `load_macro_data()` to load all four required series from the `data/` folder, enforce monthly `DatetimeIndex`, and verify monotonicity.


In [ ]:
def load_macro_data():
    """
    Loads US Headline CPI (CPIAUCSL), WTI Crude Oil (MCOILWTICO),
    Retail Sales (RSAFS), and Unemployment Rate (UNRATE).
    
    Returns:
        dict: {'cpi': Series, 'oil': Series, 'rsafs': Series, 'unrate': Series}
    """
    def _read_fred(filename, col_name):
        filepath = DATA_DIR / filename
        df = pd.read_csv(filepath)
        date_col = 'observation_date' if 'observation_date' in df.columns else 'DATE'
        val_col = col_name if col_name in df.columns else df.columns[1]
        df['DATE'] = pd.to_datetime(df[date_col])
        df = df.set_index('DATE').sort_index()
        s = pd.to_numeric(df[val_col], errors='coerce').dropna()
        return s.asfreq('MS')

    return {
        'cpi': _read_fred('CPIAUCSL.csv', 'CPIAUCSL'),
        'oil': _read_fred('MCOILWTICO.csv', 'MCOILWTICO'),
        'rsafs': _read_fred('RSAFS.csv', 'RSAFSNA' if 'RSAFSNA' in pd.read_csv(DATA_DIR / 'RSAFS.csv').columns else 'RSAFS'),
        'unrate': _read_fred('UNRATE.csv', 'UNRATE'),
    }

# Quick verification
macro_data = load_macro_data()
for k, v in macro_data.items():
    print(f"{k.upper()}: {len(v)} observations from {v.index[0].date()} to {v.index[-1].date()}")


## 📈 Task 2: Deterministic Trends, Fourier Harmonics & AICc
Implement `build_trend_and_fourier_features()` and `compute_aicc()`.


In [ ]:
def compute_aicc(ss_res: float, n_obs: int, n_params: int) -> float:
    """
    Computes the finite-sample corrected Akaike Information Criterion (AICc).
    
    Formula:
        AIC = n * ln(ss_res / n) + 2 * (k + 2)
        AICc = AIC + 2*(k+2)*(k+3) / (n - k - 3)
    where k is the number of slope predictors (excluding intercept & sigma^2).
    """
    p_total = n_params + 2  # k slope coefficients + intercept + error variance
    aic = n_obs * np.log(ss_res / n_obs) + 2 * p_total
    denom = n_obs - n_params - 3
    if denom <= 0:
        return np.inf
    correction = (2 * p_total * (p_total + 1)) / denom
    return float(aic + correction)


def build_trend_and_fourier_features(series: pd.Series, trend_type: str = "linear", K: int = 1, knot_date: pd.Timestamp = None) -> pd.DataFrame:
    """
    Constructs linear/piecewise trend features and Fourier harmonic terms.
    """
    n = len(series)
    t = np.arange(1, n + 1, dtype=float)
    feat = pd.DataFrame(index=series.index)
    feat['trend'] = t

    if trend_type == "piecewise" and knot_date is not None:
        knot_idx = np.where(series.index >= knot_date)[0]
        tau = knot_idx[0] + 1 if len(knot_idx) > 0 else n
        feat['spline_knot'] = np.maximum(0.0, t - tau)

    # Fourier harmonic pairs for annual periodicity m=12
    m = 12.0
    for k in range(1, K + 1):
        feat[f'sin_{k}'] = np.sin(2.0 * np.pi * k * t / m)
        feat[f'cos_{k}'] = np.cos(2.0 * np.pi * k * t / m)

    return feat


## ⏱️ Task 3: Causal Lag Alignment & The Leak-Audit Checker
Implement `create_lagged_matrix()` and the defensive assertion `verify_causal_alignment()`.


In [ ]:
def create_lagged_matrix(target: pd.Series, predictor: pd.Series, lags: list = [1, 2, 3], drop_na: bool = True):
    """
    Creates a dynamic feature table with strictly causal lagged predictors.
    """
    for l in lags:
        if l < 1:
            raise ValueError(f"All lag orders must be strictly positive (>=1), got lag={l}")

    df = pd.DataFrame({"target": target})
    for l in lags:
        df[f"{predictor.name}_lag{l}"] = predictor.shift(l)

    if drop_na:
        df = df.dropna()

    X = df.drop(columns=["target"])
    y = df["target"]
    return X, y


def verify_causal_alignment(X: pd.DataFrame, y: pd.Series) -> bool:
    """
    Defensive assertion: Audits feature table for temporal look-ahead leakage.
    Raises AssertionError if future information is detected in current rows.
    """
    assert X.index.equals(y.index), "Feature index and target index must be strictly identical"
    assert X.index.is_monotonic_increasing, "Timestamps must be monotonically sorted"

    # Verify no feature column contains future values by checking lag names and correlations
    for col in X.columns:
        if "future" in col.lower() or "lead" in col.lower() or "shift(-" in col.lower():
            raise AssertionError(f"Catastrophic look-ahead detected in column name: {col}")

    return True


## 🔬 Task 4: Residual Diagnostics & Autocorrelation Tests
Implement `run_durbin_watson()` and `run_breusch_godfrey_test()`.


In [ ]:
def run_durbin_watson(residuals: np.ndarray) -> float:
    """
    Computes the Durbin-Watson statistic for first-order serial correlation.
    """
    e = np.asarray(residuals).flatten()
    diff_e = np.diff(e)
    dw = np.sum(diff_e ** 2) / np.sum(e ** 2)
    return float(dw)


def run_breusch_godfrey_test(residuals: np.ndarray, X: np.ndarray, nlags: int = 4) -> dict:
    """
    Executes the Breusch-Godfrey Lagrange Multiplier test for higher-order serial correlation.
    """
    e = np.asarray(residuals).flatten()
    n = len(e)
    # Create lag matrix of residuals
    e_lags = np.zeros((n, nlags))
    for l in range(1, nlags + 1):
        e_lags[l:, l - 1] = e[:-l]

    # Auxiliary regression: e on X and residual lags
    X_aux = np.column_stack([X, e_lags])
    # Drop initial nlags burn-in rows
    X_aux_eff = X_aux[nlags:]
    e_eff = e[nlags:]

    beta_aux = np.linalg.lstsq(X_aux_eff, e_eff, rcond=None)[0]
    e_pred = X_aux_eff @ beta_aux
    ss_tot = np.sum((e_eff - np.mean(e_eff)) ** 2)
    ss_res_aux = np.sum((e_eff - e_pred) ** 2)
    r2_aux = 1.0 - (ss_res_aux / ss_tot) if ss_tot > 0 else 0.0

    lm_stat = float((n - nlags) * r2_aux)
    p_val = float(1.0 - stats.chi2.cdf(lm_stat, df=nlags))

    return {
        "lm_stat": lm_stat,
        "p_value": p_val,
        "has_serial_correlation": bool(p_val < 0.05)
    }


## 🤝 Task 5: Forecast Combinations & Walk-Forward Validation
Implement `bates_granger_combination()` and `walk_forward_ensemble_evaluation()`.


In [ ]:
def bates_granger_combination(forecasts: dict, historical_errors: dict, method: str = "equal"):
    """
    Combines point forecasts using Simple Equal (1/M) or Inverse-Variance weighting.
    """
    keys = sorted(forecasts.keys())
    M = len(keys)
    f_vec = np.array([forecasts[k] for k in keys])

    if method == "equal":
        w = np.ones(M) / M
    elif method == "inverse_variance":
        mse_vec = np.array([historical_errors[k] for k in keys])
        inv_mse = 1.0 / np.maximum(mse_vec, 1e-8)
        w = inv_mse / np.sum(inv_mse)
    else:
        raise ValueError(f"Unknown combination method: {method}")

    comb_val = float(np.dot(w, f_vec))
    weights_dict = {k: float(w[i]) for i, k in enumerate(keys)}
    return comb_val, weights_dict


def walk_forward_ensemble_evaluation(target: pd.Series, predictor: pd.Series, n_splits: int = 5, train_window: int = 80, test_horizon: int = 1) -> dict:
    """
    Expanding walk-forward evaluation comparing individual models and ensembles.
    """
    n = len(target)
    split_size = (n - train_window) // n_splits

    errors = {"naive": [], "ar_p": [], "dynamic_ols": [], "ensemble_equal": [], "ensemble_inv_var": []}

    for s in range(n_splits):
        split_origin = train_window + s * split_size
        if split_origin + test_horizon > n:
            break

        y_train = target.iloc[:split_origin]
        y_test = target.iloc[split_origin:split_origin + test_horizon].values[0]
        x_train = predictor.iloc[:split_origin]

        # 1. Model: Naive
        f_naive = y_train.iloc[-1]

        # 2. Model: AR(1)
        X_ar = y_train.iloc[:-1].values.reshape(-1, 1)
        y_ar = y_train.iloc[1:].values
        beta_ar = np.linalg.lstsq(np.column_stack([np.ones(len(X_ar)), X_ar]), y_ar, rcond=None)[0]
        f_ar = beta_ar[0] + beta_ar[1] * y_train.iloc[-1]

        # 3. Model: Dynamic OLS with lag-1 oil
        X_dyn = np.column_stack([y_train.iloc[:-1].values, x_train.iloc[:-1].values])
        beta_dyn = np.linalg.lstsq(np.column_stack([np.ones(len(X_dyn)), X_dyn]), y_ar, rcond=None)[0]
        f_dyn = beta_dyn[0] + beta_dyn[1] * y_train.iloc[-1] + beta_dyn[2] * x_train.iloc[-1]

        # Ensembling
        f_dict = {"ar_p": f_ar, "dynamic_ols": f_dyn}
        # Historical MSE estimated strictly on train
        res_ar = y_ar - (beta_ar[0] + beta_ar[1] * X_ar.flatten())
        res_dyn = y_ar - (beta_dyn[0] + beta_dyn[1] * X_dyn[:, 0] + beta_dyn[2] * X_dyn[:, 1])
        err_dict = {"ar_p": np.mean(res_ar**2), "dynamic_ols": np.mean(res_dyn**2)}

        f_eq, _ = bates_granger_combination(f_dict, err_dict, method="equal")
        f_inv, _ = bates_granger_combination(f_dict, err_dict, method="inverse_variance")

        errors["naive"].append((y_test - f_naive)**2)
        errors["ar_p"].append((y_test - f_ar)**2)
        errors["dynamic_ols"].append((y_test - f_dyn)**2)
        errors["ensemble_equal"].append((y_test - f_eq)**2)
        errors["ensemble_inv_var"].append((y_test - f_inv)**2)

    return {k: float(np.sqrt(np.mean(v))) for k, v in errors.items()}


## ✍️ Task 6: Technical Interview Clinic (Syllabus Questions)

### Question 1: How do you create lagged predictors and align them correctly, and what goes wrong if you shift in the wrong direction?
**Response:**
Lagged predictors are created using `pd.Series.shift(k)` where $k \ge 1$ is a strictly positive integer. This shifts past observations down the index, so row $t$ contains information from timestamp $t-k$. 
If shifted in the wrong direction using negative integers (`shift(-k)`), future values from $t+k$ are pulled backward into the current row. This causes catastrophic look-ahead data leakage, creating near-perfect artificial $R^2$ in backtests that completely collapses in live operational deployment.

---

### Question 2: How do you choose combination weights without peeking at the test set?
**Response:**
Combination weights must be estimated strictly on the historical training set or rolling pre-origin validation folds. For inverse-variance weighting, each model's MSE is calculated purely on in-sample residuals or rolling walk-forward cross-validation splits prior to date $T$. 
At no point may the holdout test set observations or test forecast errors enter the weight estimation formula, as doing so constitutes look-ahead snooping and produces biased performance estimates.

---

### Question 3: How do you check regression residuals for autocorrelation before trusting the standard errors?
**Response:**
Regression residuals must be checked using the Durbin-Watson statistic and the Breusch-Godfrey Lagrange Multiplier (LM) test, alongside inspecting the sample ACF of residuals. 
If residuals exhibit significant serial correlation, the Gauss-Markov assumptions are violated: OLS standard errors are severely underestimated, producing artificially inflated $t$-statistics and phantom statistical significance (as in the Granger-Newbold spurious regression trap). Furthermore, autocorrelated residuals signal unexploited predictable structure that should be captured via dynamic distributed lags or ARIMA error terms.
